In [19]:
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic
import json
import re
from statistics import mean

client = Anthropic()
model = "claude-haiku-5-5" #"claude-sonnet-5-5"

In [21]:
def remove_json_markdown(response: str) -> str:
    ''' Removes the Markdown formatting from a JSON string '''

    m = re.search(r'^```json\n(.+)```$', response, re.DOTALL) #, re.MULTILINE | re.DOTALL)
    m1 = m.group(1)
    return m1


In [31]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None): #,stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 16000,
        "messages": messages
    }
    if system:
        params["system"] = system
    # if stop_sequences:
    #     params["stop_sequences"] = stop_sequences
    
    response = client.messages.create(**params)
    print(f'RESPONSE CONTENT: \n  {response.content}')
    return response.content[0].text

In [23]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""
    
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [24]:
def grade_by_model(test_case, solution): #grade_by_model(test_case, output):
    # Create evaluation prompt
    eval_prompt = f"""
    You are an expert code reviewer. Evaluate this AI-generated solution.
    
    Task: {test_case["task"]}
    Solution: {solution}
    
    Provide your evaluation as a structured JSON object with:
    - "strengths": An array of 1-3 key strengths
    - "weaknesses": An array of 1-3 key areas for improvement  
    - "reasoning": A concise explanation of your assessment
    - "score": A number between 1-10
    """
    
    messages = []
    add_user_message(messages, eval_prompt)
    # add_assistant_message(messages, "```json")

    eval_text = chat(messages) #, stop_sequences=["```"])
    eval_text = remove_json_markdown(eval_text) # remove the Markdown formatting from the JSON string
    return json.loads(eval_text) # deserialize the JSON string into a Python dictionary

In [25]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    # TODO - Grading
    # score = 10
    model_grade = grade_by_model(test_case, output) 
    score = model_grade["score"] 

    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }

In [27]:
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results]) 
    print(f"Average score: {average_score}") 
        
    return results

In [32]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

RESPONSE CONTENT: 
  [TextBlock(citations=None, text='2 + 2 = **4**', type='text')]
RESPONSE CONTENT: 
  [ThinkingBlock(signature='CAQSiQcKEAgSGAI4AUIIdGhpbmtpbmcSDFhovCELzLtJYkKpVBoM9mz4wgs3Rp6/CjvEIjAryHo3IKyyocbhKKOcjn8nXx5alDeqJqJjdIgdgzVLpia46vWlfa5TxoB5osW4V/cqpgbXV/Gmn4okgQbPsKHvfk6RjzlENXzQdlWmMIvpebFe6HwS9f2b63x5ip3tvoHO6cSTFJXm8YJUOq9KMcLG2qzBzPjgaE5Nm5EZs44nc3rS/WeDBf558UdICtkmx0VjfnH3EKSmD+OUjk7ycTHEel8IoKa6Wm01OoICh4KJRJ2et0zWleG9gMnASjLxegocxBRtYSpD6DX6djTTAmjk8SYkk+TAGOVUonmRRt7cPKnG9ogx4pS1IuMZJm40jNh0swzxRSVDsU/x7+H+R/YVD17JwZkv9p7LYrcfPGCf6il3JcHSsvycyjab7ugj1uMZpLfTOOn9TLbmF+lcFrTSrsnGdifx0RnwyBNTMvsmYbwKJjfRRbkayJXDDZhSmx6OGYvyRpy6qwPJVRF4hNOk0RKFYw9yCZ15dcOUmWTKos9V3G3i+J2XOWF03gUYe7ABGEioUKOpBtNIofudMoMhY/B/k84FDhUsLflDzXmXPlep2zIlGUbSpAFqMEgoDk9ky3gayJul6dqomUkISoFK6GtBZZDCeV+v+l69vwsufzXpZugsQQJiix6CQlU8bzDUa+8RtIuriuiF03g2HxZllL+Bt5hFJ40ujvVcdqH7GaSCqd16aHpLHVXHzrCI/FM0vIIV/J+fwYUbhEGdzB3HCO/irhxMJd5GvP3hMtHGvdu1SqQjw2CcSqc4a94np0rV2iKNd7P9xhmJMWmtkIRIfe7UpWGPd

AttributeError: 'ThinkingBlock' object has no attribute 'text'